# Phenology baseline — output

Runs `scripts/build_phenology_stats.py` and visualizes what it produces
(`data/count/species_doy_statistics.json`) for every modelled species: the day-of-year
count-rate distribution and the hour-of-day activity profile that `src.phenology.Phenology`
uses as the evaluation baseline, and that defileViz renders as its uncertainty band.

Everything below reads that output through `Phenology` — the same class every other
consumer (the test report, `val/skill_vs_phenology`) uses — rather than re-parsing the
JSON by hand, so nothing about the fit or the statistics is duplicated here. For exploring
or tuning the fit itself (spline counts, regularization, raw ratio samples before
smoothing), use `PhenologyBuilder` from `scripts/build_phenology_stats.py` directly — that
is a different job from this notebook's, which is only to look at what the script
actually produced.

In [ ]:
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils

rootutils.setup_root(".", indicator=".project-root", pythonpath=True)

from scripts.build_phenology_stats import species_from_experiments
from src.phenology import PHENOLOGY_FILE, RATIO_HOURS, Phenology

DATA_DIR = "../data"
CONFIGS_DIR = "../configs"

## Run the script

Regenerates `data/count/species_doy_statistics.json` for every species in `configs/experiment/*.yaml` — a couple of minutes, one GAM fit per species.

In [ ]:
result = subprocess.run(
    [sys.executable, "scripts/build_phenology_stats.py"],
    cwd="..",
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError(f"build_phenology_stats.py exited {result.returncode}")

## Load the output

Through `Phenology.load` — the exact path `src/metrics.py` (which re-exports it from `src/phenology.py`) and the test report use.

In [ ]:
species_list = species_from_experiments(CONFIGS_DIR)
phenologies = {s: Phenology.load(DATA_DIR, s) for s in species_list}

# A couple of fields (trektellen_species_id, count_observations) live in the raw file but
# not on `Phenology` -- it only loads what `src/metrics.py` itself consumes (defined in `src/phenology.py`). Read them
# straight from the JSON rather than adding them to `Phenology` just for this table.
with open(f"{DATA_DIR}/{PHENOLOGY_FILE}") as f:
    raw_by_species = {r["species"]: r for r in json.load(f)}

print(f"{len(phenologies)} species loaded")

## Summary table

One row per species: how much data backs the baseline, and its coarse phenology (peak day, peak rate).

In [ ]:
def summarize(species):
    p = phenologies[species]
    raw = raw_by_species[species]
    peak = int(np.argmax(p.mean))
    return {
        "species": species,
        "trektellen_id": raw["trektellen_species_id"],
        "observation_days": int(np.sum(raw["count_observations"])),
        "peak_doy": int(p.doy[peak]),
        "peak_mean_rate": float(p.mean[peak]),
        "season_mean_rate": float(np.mean(p.mean)),
    }


summary = pd.DataFrame(summarize(s) for s in species_list).sort_values(
    "season_mean_rate", ascending=False
).reset_index(drop=True)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = summary.sort_values("observation_days")
ax.barh(order["species"], order["observation_days"], color="tab:blue")
ax.set_xlabel("Observation-days backing the baseline")
ax.set_title("How much data each species' phenology baseline rests on")
plt.tight_layout()
plt.show()

## Day-of-year distribution, per species

Mean daily rate (`Phenology.mean`) and the p20-p80 band (`Phenology.quantile`), one panel
per species, ordered by season-mean rate (busiest species first).

In [ ]:
species_order = summary["species"].tolist()

fig, axes = plt.subplots(len(species_order), 1, figsize=(9, 1.9 * len(species_order)), sharex=True)
for ax, species in zip(axes, species_order):
    p = phenologies[species]
    ax.plot(p.doy, p.mean, color="black", lw=1.2, label="mean")
    ax.fill_between(
        p.doy, p.quantile(p.doy, 20), p.quantile(p.doy, 80), color="gray", alpha=0.35, label="p20-p80"
    )
    ax.set_ylabel(species, rotation=0, ha="right", va="center", fontsize=8)
    ax.set_yticks([])

axes[0].legend(loc="upper right", fontsize=7)
axes[-1].set_xlabel("Day of year")
fig.suptitle("Day-of-year count-rate distribution (phenology baseline)", y=1.0)
plt.tight_layout()
plt.show()

## Hour-of-day activity, per species

The fitted `ratio` surface (`Phenology.ratio`, hourly rate / that day's rate) — the shape
`val/skill_vs_phenology` and the report's diurnal-shape panel compare predictions against.

In [ ]:
fig, axes = plt.subplots(
    len(species_order), 1, figsize=(9, 1.3 * len(species_order)), sharex=True, constrained_layout=True
)
im = None
for ax, species in zip(axes, species_order):
    p = phenologies[species]
    im = ax.imshow(
        p.ratio.T,
        aspect="auto",
        origin="lower",
        extent=[p.doy[0], p.doy[-1], RATIO_HOURS[0], RATIO_HOURS[-1] + 1],
        cmap="viridis",
    )
    ax.set_ylabel(species, rotation=0, ha="right", va="center", fontsize=8)
    ax.set_yticks([RATIO_HOURS[0], RATIO_HOURS[-1] + 1])

axes[-1].set_xlabel("Day of year")
fig.suptitle("Hour-of-day activity ratio (hourly rate / daily rate)")
fig.colorbar(im, ax=axes, label="ratio", shrink=0.6)
plt.show()

## Next steps

- Exploring or tuning the GAM fit itself (spline counts, regularization, raw pre-smoothing
  ratio samples) belongs with `PhenologyBuilder` in `scripts/build_phenology_stats.py`, not
  here — that script's module docstring has the details.
- Skill scores computed against this baseline are in every species' test report
  (`src/plots/report.py`) and logged every validation epoch as `val/skill_vs_phenology`
  (`src/models/defile_module.py`).